In [8]:
import pandas as pd
import numpy as np
import pickle
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

print("Libraries loaded successfully")

Libraries loaded successfully


In [9]:
np.random.seed(42)
n_samples = 15000

# Event types: 0=concert, 1=cricket match, 2=festival, 3=protest, 4=holiday
event_type     = np.random.randint(0, 5, n_samples)

# Event size: 0=small, 1=medium, 2=large
event_size     = np.random.randint(0, 3, n_samples)

# Distance from zone to event in km
distance_km    = np.random.uniform(0.1, 10.0, n_samples)

# Hours until event starts (0 = happening now)
hours_to_event = np.random.uniform(0, 24, n_samples)

# Day of week
day_of_week    = np.random.randint(0, 7, n_samples)

# Build dataframe
events_df = pd.DataFrame({
    'event_type':     event_type,
    'event_size':     event_size,
    'distance_km':    distance_km,
    'hours_to_event': hours_to_event,
    'day_of_week':    day_of_week
})

print("Synthetic dataset created")
print("Shape:", events_df.shape)
events_df.head()

Synthetic dataset created
Shape: (15000, 5)


,event_type,event_size,distance_km,hours_to_event,day_of_week
0,3,0,5.255379,23.511159,3
1,4,0,9.621263,9.581269,2
2,2,2,2.916165,7.604874,6
3,4,2,7.640355,10.043234,4
4,4,0,5.790285,2.779150,1


In [10]:
def calculate_multiplier(row):
    base = 1.0

    # Event size impact
    size_boost = {0: 0.2, 1: 0.8, 2: 2.0}
    base += size_boost[row['event_size']]

    # Event type impact
    type_boost = {0: 0.3, 1: 1.0, 2: 1.5, 3: 0.5, 4: 0.2}
    base += type_boost[row['event_type']]

    # Distance decay — closer = bigger impact
    distance_factor = max(0, 1 - (row['distance_km'] / 10))
    base *= (1 + distance_factor)

    # Time factor — impact strongest 2 hours before event
    if row['hours_to_event'] <= 2:
        base *= 1.3
    elif row['hours_to_event'] <= 6:
        base *= 1.1

    # Weekend boost
    if row['day_of_week'] >= 5:
        base *= 1.1

    return round(base, 2)

events_df['multiplier'] = events_df.apply(calculate_multiplier, axis=1)

print("Multiplier stats:")
print(events_df['multiplier'].describe())
print("\nSample multipliers:")
print(events_df[['event_type','event_size','distance_km','multiplier']].head(10))

Multiplier stats:
count    15000.000000
mean         4.303577
std          1.709736
min          1.400000
25%          2.960000
50%          4.030000
75%          5.390000
max         12.800000
Name: multiplier, dtype: float64

Sample multipliers:
   event_type  event_size  distance_km  multiplier
0           3           0     5.255379        2.51
1           4           0     9.621263        1.45
2           2           2     2.916165        8.46
3           4           2     7.640355        3.96
4           4           0     5.790285        2.19
5           1           2     7.037905        5.70
6           2           0     4.236393        4.26
7           2           2     2.864155        7.71
8           2           1     5.506647        4.78
9           4           0     3.811917        2.27


In [11]:
features = ['event_type', 'event_size', 'distance_km', 
            'hours_to_event', 'day_of_week']

X = events_df[features]
y = events_df['multiplier']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)
print("XGBoost model trained successfully")

XGBoost model trained successfully


In [12]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print("MAE:", round(mae, 4))
print("R2 Score:", round(r2, 4))
print("Accuracy:", round(r2 * 100, 2), "%")

MAE: 0.0306
R2 Score: 0.999
Accuracy: 99.9 %


In [13]:
# Test 1 — Large festival very close (should be HIGH multiplier > 3.0)
large_local = pd.DataFrame([{
    'event_type': 2,      # festival
    'event_size': 2,      # large
    'distance_km': 0.3,   # very close
    'hours_to_event': 1,  # starting soon
    'day_of_week': 6      # Sunday
}])
pred1 = model.predict(large_local)[0]
print("Large local festival multiplier:", round(pred1, 2),
      "✅ HIGH" if pred1 > 3.0 else "❌ Should be > 3.0")

# Test 2 — Small distant event (should be LOW multiplier ~1.0)
small_distant = pd.DataFrame([{
    'event_type': 0,      # concert
    'event_size': 0,      # small
    'distance_km': 9.5,   # very far
    'hours_to_event': 20, # far away in time
    'day_of_week': 2      # Wednesday
}])
pred2 = model.predict(small_distant)[0]
print("Small distant event multiplier:", round(pred2, 2),
      "✅ LOW" if pred2 < 2.0 else "❌ Should be ~1.0")

Large local festival multiplier: 12.56 ✅ HIGH
Small distant event multiplier: 1.56 ✅ LOW


In [14]:
os.makedirs('../models/saved', exist_ok=True)

with open('../models/saved/event_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print("Event model saved to models/saved/event_model.pkl")

Event model saved to models/saved/event_model.pkl
